# 🛠️ Chapter 06-01b: Hands-On — Build a ReAct Agent from Scratch

---

## Build an Agent That Thinks and Acts! 🤖

By the end:
- ✅ Built the ReAct loop manually (understand the internals)
- ✅ Used LangChain's create_react_agent
- ✅ Added memory to an agent
- ✅ Built a multi-tool research agent

Let's build! 🚀

---

## 📦 Setup

In [ ]:
# !pip install langchain-openai langchain-core langchain python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ Ready!")

---

## 🧪 Step 1: Define Tools

In [ ]:
import math
from datetime import datetime

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression. Input: valid Python math like '2+3*4' or 'math.sqrt(16)'."""
    try:
        allowed = set('0123456789+-*/.() mathsqrtpowlog')
        clean = expression.replace(' ', '')
        result = eval(expression, {"__builtins__": {}, "math": math})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def search(query: str) -> str:
    """Search the internet for current information. Use for facts, news, data."""
    # Simulated search results
    data = {
        "population": "World population is approximately 8.1 billion (2024).",
        "tallest building": "Burj Khalifa in Dubai, UAE at 828 meters (2,717 ft).",
        "python": "Python 3.12 was released in October 2023. Python 3.13 in October 2024.",
        "uae": "UAE population is approximately 9.9 million (2024). Capital: Abu Dhabi.",
    }
    for key, value in data.items():
        if key in query.lower():
            return value
    return f"Search results for '{query}': Information not found in demo database."

@tool
def get_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [calculator, search, get_time]
tool_map = {t.name: t for t in tools}

print("🔧 Tools available:")
for t in tools:
    print(f"   • {t.name}: {t.description[:60]}")

---

## 🧪 Step 2: Build ReAct Loop from Scratch

In [ ]:
# Manual ReAct agent — understand the internals!

def react_agent(question: str, max_iterations: int = 5) -> str:
    """A ReAct agent built from scratch."""
    llm_with_tools = llm.bind_tools(tools)
    
    messages = [
        SystemMessage(content="""You are a helpful assistant with access to tools.
Use tools when you need real data or calculations.
Think step by step. Use tools one at a time."""),
        HumanMessage(content=question)
    ]
    
    for i in range(max_iterations):
        print(f"\n--- Iteration {i+1} ---")
        
        # LLM decides what to do
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # If no tool calls → final answer
        if not response.tool_calls:
            print(f"💬 Final Answer")
            return response.content
        
        # Execute each tool call
        for tc in response.tool_calls:
            print(f"🔧 Action: {tc['name']}({tc['args']})")
            
            tool_fn = tool_map[tc['name']]
            result = tool_fn.invoke(tc['args'])
            
            print(f"👁️ Observation: {result}")
            
            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tc['id']
            ))
    
    return "Max iterations reached."

# Test with a multi-step question
print("❓ What's the population of the country with the tallest building?")
answer = react_agent("What's the population of the country with the tallest building?")
print(f"\n✅ {answer}")

In [ ]:
# Test with a math + search question
print("❓ What is the square root of the world population?")
answer = react_agent("What is the square root of the world population?")
print(f"\n✅ {answer}")

---

## 🧪 Step 3: Agent with Conversation Memory

In [ ]:
class ConversationalAgent:
    def __init__(self, tools, max_iterations=5):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)
        self.tools = {t.name: t for t in tools}
        self.max_iterations = max_iterations
        self.history = [
            SystemMessage(content="""You are a helpful assistant with memory.
Remember previous conversations. Use tools when needed.""")
        ]
    
    def chat(self, message: str) -> str:
        self.history.append(HumanMessage(content=message))
        
        for _ in range(self.max_iterations):
            response = self.llm.invoke(self.history)
            self.history.append(response)
            
            if not response.tool_calls:
                return response.content
            
            for tc in response.tool_calls:
                result = self.tools[tc['name']].invoke(tc['args'])
                self.history.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
        
        return "Reached max iterations."

# Test conversational memory
agent = ConversationalAgent(tools)

conversations = [
    "What time is it?",
    "What's 15 * 37?",
    "Search for the latest Python version.",
    "Based on what we discussed, summarize the key facts.",  # Tests memory!
]

for msg in conversations:
    print(f"👤 {msg}")
    response = agent.chat(msg)
    print(f"🤖 {response}")
    print("-" * 50)

---

## ✍️ Practice: Add a Weather Tool

**Challenge:** Add a weather lookup tool and test the agent with multi-tool questions.

In [ ]:
# YOUR CODE HERE

# TODO: Create a @tool get_weather(city: str) -> str that returns simulated weather
# TODO: Add it to the tools list
# TODO: Test: "What's the weather in Dubai and what's 32°C in Fahrenheit?"

<details>
<summary>💡 Click to see solution</summary>

```python
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    weather_data = {
        "dubai": "Dubai: 38°C, sunny, humidity 45%",
        "london": "London: 12°C, cloudy, humidity 80%",
        "tokyo": "Tokyo: 22°C, partly cloudy, humidity 60%",
        "new york": "New York: 18°C, clear, humidity 55%",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

all_tools = [calculator, search, get_time, get_weather]
weather_agent = ConversationalAgent(all_tools)
print(weather_agent.chat("What's the weather in Dubai and what's 38°C in Fahrenheit?"))
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 ReAct loop: LLM → tool_calls? → execute → repeat until done. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Always set max_iterations to prevent infinite loops.           │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Tool descriptions are crucial — the LLM reads them!           │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Conversation history → persistent agent memory.                │
├─────────────────────────────────────────────────────────────────────┤
│  📌 You built an agent from scratch. LangGraph automates this.     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Tool Calling & Function Calling Deep Dive!

---

### 🎉 You Built an AI Agent!
Think → Act → Observe — the core loop powering modern AI agents. 🚀